# Bronze — VRA (Voo Regular Ativo)
Lê os 12 CSVs mensais do volume `voebem.bronze.arquivos/vra/` e materializa `voebem.bronze.vra`.

Regras da camada Bronze:
- **nada de tipagem**: tudo string, exatamente como veio do arquivo;
- **nada de filtro**: nenhuma linha é descartada;
- **colunas de auditoria**: de qual arquivo veio e quando foi ingerido;
- **idempotente**: rodar duas vezes não duplica.

In [2]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

# Leitura
Quatro opções carregam quatro problemas do arquivo:
| **opção** | **resolve** |
|   ---     |   ---       |
| `sep = ";"`  | separador brasileiro, não vírgula |
| `skipRows = 1` | a 1º linha é `atualizada em: <data>`, não o cabeçalho - e o BOM `EF` `BB` `BF` mora nela, some junto |
| `header = true` | a 2º linha (a primeira que sobra) é o cabeçalho de verdade |
| `inferSchema` **desligado** (default) | bronze não tipa: tudo chega como `string` |

In [ ]:
bruto = (
    spark.read.format("csv")
        .option("sep", ";")
        .option("header", True)
        .option("skipRows", 1)
        .option("quote", '"')
        .option("escape", '"')
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .load(CAMINHO)
)

print("colunas lidas do arquivo:")

for c in bruto.columns:
    print(f" {c!r}")

# Nome de coluna: o Delta não aceita espaço

`ICAO Empresa Aérea` é um nome de coluna válida em CSV e **inválido** em Delta — espaço está na lista de caracteres proibidos ( `,;{}()\n\t=`).

Então normalizamos o **nome**. Repare que isso não fere a regra da bronze: o que a bronze preserva é o **valor** e a **granularidade**, não a grafia do cabeçalho. Nenhuma coluna é somada, removida, filtrada ou convertida.

O mapa fica explícito no código — nada de `regexp_replace` mágico, para que a correspondência com o arquivo original seja auditável.

In [ ]:
RENOMEAR = {
    "ICAO Empresa Aérea": "icao_empresa",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_origem",
    "ICAO Aeródromo Destino": "icao_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa",
}

faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Coluna esperada não encontrada no CSV: {faltando}"

renomeado = bruto.select(
    *[F.col(f"`{origem}`").cast("string").alias(novo) for origem, novo in RENOMEAR.items()]
)

# Auditoria

Duas colunas que o arquivo não tem e a tabela precisa ter: `_arquivo_origem` (de qual CSV a linha veio — `_metadata` é uma coluna oculta que o Spark expõe em qualquer leitura de arquivo) e `_ingerido_em`.

In [ ]:
bronze = renomeado.withColumn("_arquivo_origem", F.col("_metadata.file_name")).withColumn("_ingerido_em", F.current_timestamp())

# Escrita idempotente

Estratégia: *full refresh determinístico* — `mode("overwrite")` sobre o conjunto inteiro de arquivos.

Por que essa e não um `append` com deduplicação:
1. A fonte é *imutável e completa*: o volume tem os 12 arquivos do mês fechado, e a ANAC republica o mês inteiro quando corrige algo. A entrada define o estado final — logo o destino pode ser derivado inteiro dela.
2. `append` exigiria uma chave de negócio para deduplicar. O VRA *não tem chave natural única* (o mesmo voo pode repetir legitimamente na mesma data — veja o código DI "Estapa de Voo Duplicada"). Deduplicar no bronze seria decidir regra de negócio na camada errada.
3. `overwrite` no Detal é * atômico* : ou a versão nova aparece inteira, ou a antiga continua valendo. Ninguém lê tabela pela metade.
4. O histórico não se perde: cada `overwrite` gera uma versão nova no logo do Delta, e a anterior continua acessível por time travel (marco-04).

O que muda entre duas execuções: só *_ingerido_em*. O *conjunto de linhas* é idêntico — é isso que a validação prova.

In [ ]:
(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

In [ ]:
spark.sql(f"""
          COMMENT ON TABLE {TABELA} IS
          'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026).
           Dado bruto: todas as colunas string, nenhuma linha descartada.
           Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/.'
          """)

In [ ]:
display(
    spark.sql(f"""
              SELECT _arquivo_origem, COUNT(*) AS linhas, MAX (_ingerido_em) AS ingerido_em
              FROM {TABELA}
              GROUP BY _arquivo_origem
              ORDER BY _arquivo_origem
              """)
)